# v2 · Fase 2 · paso 2 — Entrenamiento YOLOv8

## Configuración

| Parámetro | Valor | Razón |
|-----------|-------|-------|
| Modelo | `yolov8n.pt` (~6 MB) | Tamaño mínimo, mejor para móvil. |
| `imgsz` | **416** | ≈2× más rápido que 640 en móvil; pérdida marginal de mAP. |
| Épocas | 200 | Suficiente con `patience=50` para early stopping. |
| `patience` | 50 | Lecciones v1: dataset chico requiere paciencia alta. |
| `batch` | 8 | Estable en MPS sin saturar memoria. |
| Augmentations | Conservadoras | Sin mosaic ni mixup (lección v1). |
| Device | MPS (auto) | Apple Silicon con `PYTORCH_ENABLE_MPS_FALLBACK=1`. |

## Salidas
- `runs/v2/coins_v2/` — pesos, métricas, gráficos.
- `model/v2/best.pt` — mejor checkpoint para exportar.
- `reports/v2/metrics_*.csv` — tablas resumen.

In [1]:
from __future__ import annotations
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from ultralytics import YOLO

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "dataset_v2").exists():
    ROOT = ROOT.parent
DATA_YAML = ROOT / "dataset_v2" / "data_train.yaml"
MODEL_DIR = ROOT / "model" / "v2"; MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORTS = ROOT / "reports" / "v2"; REPORTS.mkdir(parents=True, exist_ok=True)
RUNS_DIR = ROOT / "runs" / "v2"
RUN_NAME = "coins_v2"
IMG_SIZE = 416

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"DATA   = {DATA_YAML}")
print(f"DEVICE = {DEVICE}")
print(f"RUN    = {RUNS_DIR / 'detect' / RUN_NAME}")
print(f"IMGSZ  = {IMG_SIZE}")

DATA   = /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/dataset_v2/data_train.yaml
DEVICE = mps
RUN    = /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/runs/v2/detect/coins_v2
IMGSZ  = 416


## 1. Entrenamiento

In [2]:
prev = RUNS_DIR / "detect" / RUN_NAME
if prev.exists():
    shutil.rmtree(prev)

model = YOLO("yolov8n.pt")
results = model.train(
    data=str(DATA_YAML),
    epochs=200,
    patience=50,
    imgsz=IMG_SIZE,
    batch=8,
    device=DEVICE,
    project=str(RUNS_DIR / "detect"),
    name=RUN_NAME,
    exist_ok=True,
    seed=42,
    deterministic=True,
    plots=True,
    verbose=True,
    # Augmentations conservadoras (lección v1: mosaic+mixup matan datasets chicos)
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
    degrees=5.0, translate=0.05, scale=0.3,
    fliplr=0.5, flipud=0.0,
    mosaic=0.0, mixup=0.0,
    erasing=0.0,
)
print("Entrenamiento finalizado.")

Ultralytics 8.4.47 🚀 Python-3.12.7 torch-2.11.0 MPS (Apple M3 Pro)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/dataset_v2/data_train.yaml, degrees=5.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=coins_v2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=50, perspective=0.0, pl

Overriding model.yaml nc=80 with nc=5



                   from  n    params  module                                       arguments                     


  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 


...

                    50          3          3          1      0.965      0.995      0.896


                   500          2          6      0.955          1      0.995      0.895


Speed: 0.3ms preprocess, 145.3ms inference, 0.0ms loss, 64.3ms postprocess per image


Results saved to /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/runs/v2/detect/coins_v2


Entrenamiento finalizado.


## 2. Persistencia del mejor checkpoint

In [3]:
best_pt = RUNS_DIR / "detect" / RUN_NAME / "weights" / "best.pt"
last_pt = RUNS_DIR / "detect" / RUN_NAME / "weights" / "last.pt"
assert best_pt.exists(), f"No se encontró {best_pt}"
shutil.copy2(best_pt, MODEL_DIR / "best.pt")
if last_pt.exists():
    shutil.copy2(last_pt, MODEL_DIR / "last.pt")
print("Pesos copiados a:", MODEL_DIR)
for p in sorted(MODEL_DIR.iterdir()):
    print("  -", p.name, f"({p.stat().st_size / 1024 / 1024:.1f} MB)")

Pesos copiados a: /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/model/v2
  - best.pt (5.9 MB)
  - last.pt (5.9 MB)


## 3. Métricas finales sobre VALID y TEST

In [4]:
best_model = YOLO(str(MODEL_DIR / "best.pt"))

def metrics_to_row(m, split):
    return {
        "split": split,
        "precision": float(m.box.mp),
        "recall": float(m.box.mr),
        "mAP@0.5": float(m.box.map50),
        "mAP@0.5:0.95": float(m.box.map),
    }

val_metrics = best_model.val(
    data=str(DATA_YAML), split="val", device=DEVICE, imgsz=IMG_SIZE,
    project=str(RUNS_DIR / "detect"), name=f"{RUN_NAME}_val", exist_ok=True, plots=True,
)
test_metrics = best_model.val(
    data=str(DATA_YAML), split="test", device=DEVICE, imgsz=IMG_SIZE,
    project=str(RUNS_DIR / "detect"), name=f"{RUN_NAME}_test", exist_ok=True, plots=True,
)

df_metrics = pd.DataFrame([
    metrics_to_row(val_metrics, "valid"),
    metrics_to_row(test_metrics, "test"),
]).set_index("split")
df_metrics.to_csv(REPORTS / "metrics_summary.csv")
df_metrics.round(4)

Ultralytics 8.4.47 🚀 Python-3.12.7 torch-2.11.0 MPS (Apple M3 Pro)


Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1677.4±109.2 MB/s, size: 71.5 KB)


val: Scanning /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/dataset_v2/valid/labels.cache... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 4.2Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 18.0it/s 0.1s

...

                    50          3          3      0.971          1      0.995      0.852


                   500          3          4      0.881          1      0.995       0.92


Speed: 0.3ms preprocess, 3.5ms inference, 0.0ms loss, 34.7ms postprocess per image


Results saved to /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/runs/v2/detect/coins_v2_test


,precision,recall,mAP@0.5,mAP@0.5:0.95
split,,,,
valid,0.9487,0.993,0.995,0.8950
test,0.9216,1.000,0.995,0.8999


## 4. Métricas por clase (sobre VALID)

In [5]:
names = best_model.names
p_cls = val_metrics.box.p
r_cls = val_metrics.box.r
ap50 = val_metrics.box.ap50
ap = val_metrics.box.ap.mean(axis=1) if val_metrics.box.ap.ndim > 1 else val_metrics.box.ap
ids = val_metrics.box.ap_class_index

df_cls = pd.DataFrame({
    "class": [names[int(i)] for i in ids],
    "precision": np.round(p_cls, 4),
    "recall":    np.round(r_cls, 4),
    "mAP@0.5":   np.round(ap50, 4),
    "mAP@0.5:0.95": np.round(ap, 4),
})
df_cls.to_csv(REPORTS / "metrics_per_class_valid.csv", index=False)
df_cls

,class,precision,recall,mAP@0.5,mAP@0.5:0.95
0,100,0.9631,1.0000,0.995,0.8756
1,1000,0.9440,1.0000,0.995,0.9016
2,200,0.8815,1.0000,0.995,0.9065
3,50,1.0000,0.9652,0.995,0.8960
4,500,0.9549,1.0000,0.995,0.8955


## 5. Inferencia de ejemplo sobre TEST con desglose monetario

La clase `modenas` (id=5) suma 0 COP — se considera moneda genérica de denominación desconocida.

In [6]:
import yaml
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
_names = cfg["names"]
if isinstance(_names, dict):
    _names = [_names[i] for i in sorted(_names)]
VALUE_COP = {i: (int(n) if str(n).isdigit() else 0) for i, n in enumerate(_names)}

test_dir = ROOT / "dataset_v2" / "test" / "images"
test_imgs = sorted(test_dir.iterdir())
n = len(test_imgs)
cols = 2; rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(12, 5 * rows))
axes = np.array(axes).reshape(-1)
for ax, img_path in zip(axes, test_imgs):
    pred = best_model.predict(
        source=str(img_path), conf=0.25, imgsz=IMG_SIZE, device=DEVICE, verbose=False,
    )[0]
    counts = {}
    total_cop = 0
    for cls_id in pred.boxes.cls.cpu().numpy().astype(int):
        nm = pred.names[int(cls_id)]
        counts[nm] = counts.get(nm, 0) + 1
        total_cop += VALUE_COP.get(int(cls_id), 0)
    annotated = pred.plot()
    ax.imshow(annotated[:, :, ::-1])
    summary = " · ".join(f"{k}: {v}" for k, v in sorted(counts.items())) or "(sin detecciones)"
    ax.set_title(f"{img_path.name[:30]}\n{summary}\nTOTAL = {total_cop} COP", fontsize=9)
    ax.axis("off")
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(REPORTS / "test_inference.png", dpi=120)
plt.show()

<Figure size 1200x1000 with 4 Axes>

## 6. Resumen y siguientes pasos

Al terminar este notebook:
- `model/v2/best.pt` — modelo listo para exportar a ONNX en Fase 3.
- `reports/v2/metrics_*.csv` — métricas reportables en la tesis.
- `runs/v2/detect/coins_v2/` — curvas de pérdida, matriz de confusión, predicciones del split valid (`val_batch0_pred.jpg`).

Próximos pasos:
- Fase 3: re-exportar a ONNX con `imgsz=416` y reemplazar `app/public/models/coin-detector.onnx`.
- Validar latencia en la app (objetivo: ≥10 FPS en gama media).
- Si la latencia sigue insuficiente, considerar quantización INT8.